# Lab 2 - Part III: Nâng cao (Advanced Filters)

**Họ tên:** [...] &nbsp;&nbsp;|&nbsp;&nbsp; **MSSV:** [...] &nbsp;&nbsp;|&nbsp;&nbsp; **Lớp:** CN22H

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

image = cv2.imread('../input/anh.jpg')
image = image[:, :, ::-1]  # BGR -> RGB
print(f'Image loaded: {image.shape}, dtype={image.dtype}')

In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(image)
plt.title('Original Image')
plt.axis('off')
plt.tight_layout()
plt.show()

---
## 1. Phát hiện cạnh Sobel (Sobel Edge Detection)
- Chuyển ảnh sang grayscale
- Tích chập với 2 kernel đạo hàm 3x3:
  - **Sobel X**: phát hiện cạnh dọc
  - **Sobel Y**: phát hiện cạnh ngang
- Gradient magnitude: $G = \sqrt{G_x^2 + G_y^2}$
- Dùng `cv2.CV_32F` để giữ dấu giá trị âm trong đạo hàm

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

sobel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

gx = cv2.filter2D(gray, cv2.CV_32F, sobel_x)
gy = cv2.filter2D(gray, cv2.CV_32F, sobel_y)
magnitude = np.sqrt(gx**2 + gy**2)
sobel = np.clip(magnitude, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(sobel, cmap='gray')
axes[1].set_title('Sobel Edge Detection')
axes[1].axis('off')
plt.tight_layout()
plt.show()

os.makedirs('output/part3_advanced', exist_ok=True)
cv2.imwrite('output/part3_advanced/sobel_edges.jpg', sobel)
print('[OK] Saved: output/part3_advanced/sobel_edges.jpg')

---
## 2. Kernel tùy chỉnh - Emboss (Emboss Filter)
- Kernel emboss 3x3 với trọng số chéo:
  ```
  [[-2, -1,  0],
   [-1,  1,  1],
   [ 0,  1,  2]]
  ```
- Tạo hiệu ứng ảnh nổi (emboss), làm nổi bật đường viền theo hướng chéo

In [ ]:
kernel_emboss = np.array([
    [-2, -1, 0],
    [-1,  1, 1],
    [ 0,  1, 2]
], dtype=np.float32)
emboss = cv2.filter2D(image, -1, kernel_emboss)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(emboss)
axes[1].set_title('Emboss Filter (custom kernel)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

cv2.imwrite('output/part3_advanced/emboss.jpg', cv2.cvtColor(emboss, cv2.COLOR_RGB2BGR))
print('[OK] Saved: output/part3_advanced/emboss.jpg')

---
## 3. So sánh các bộ lọc (Filter Comparison)
- Hiển thị 4 ảnh trên cùng một grid:
  - **Original** | **Mean Blur (3x3)** | **Gaussian (3x3, σ=1)** | **Sharpen**
- Dễ dàng so sánh trực quan hiệu quả của từng bộ lọc

In [ ]:
mean_blur = cv2.blur(image, (3, 3))
gaussian_blur = cv2.GaussianBlur(image, (3, 3), 1)

kernel_sharpen = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
], dtype=np.float32)
sharpen = cv2.filter2D(image, -1, kernel_sharpen)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(mean_blur); axes[1].set_title('Mean Filter (3x3)'); axes[1].axis('off')
axes[2].imshow(gaussian_blur); axes[2].set_title('Gaussian Filter (3x3)'); axes[2].axis('off')
axes[3].imshow(sharpen); axes[3].set_title('Sharpen Filter'); axes[3].axis('off')

plt.tight_layout()
plt.show()

# Save the comparison grid
fig.savefig('output/part3_advanced/filter_comparison.jpg', bbox_inches='tight', dpi=100)
plt.close(fig)
print('[OK] Saved: output/part3_advanced/filter_comparison.jpg')

---
## 4. Lọc phi tuyến tính (Nonlinear Filters)

### 4.1 Lọc song phương (Bilateral Filter)
- Bộ lọc phi tuyến tính, kết hợp cả **khoảng cách không gian** và **khoảng cách cường độ**
- Ưu điểm: giảm nhiễu nhưng vẫn **giữ được cạnh** (edge-preserving)
- Tham số: `d=9` (đường kính vùng lân cận), `sigmaColor=75`, `sigmaSpace=75`

### 4.2 Lọc trung vị (Median Filter) - lọc phi tuyến
- `cv2.medianBlur` cũng là bộ lọc phi tuyến
- So sánh với bilateral để thấy sự khác biệt

In [ ]:
# Median filter (nonlinear)
median_nl = cv2.medianBlur(image, 5)

# Bilateral filter: d=9, sigmaColor=75, sigmaSpace=75
bilateral = cv2.bilateralFilter(image, 9, 75, 75)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(median_nl)
axes[1].set_title('Median Filter (k=5)')
axes[1].axis('off')
axes[2].imshow(bilateral)
axes[2].set_title('Bilateral Filter (d=9, σ=75)')
axes[2].axis('off')
plt.tight_layout()
plt.show()

cv2.imwrite('output/part3_advanced/median_nonlinear.jpg', cv2.cvtColor(median_nl, cv2.COLOR_RGB2BGR))
cv2.imwrite('output/part3_advanced/bilateral_filtered.jpg', cv2.cvtColor(bilateral, cv2.COLOR_RGB2BGR))
print('[OK] Saved: output/part3_advanced/median_nonlinear.jpg')
print('[OK] Saved: output/part3_advanced/bilateral_filtered.jpg')

---
## Tổng kết Part III

| STT | Tác vụ | Phương pháp | Output file |
|-----|--------|-------------|-------------|
| 1 | Phát hiện cạnh Sobel | Gradient X/Y + magnitude | `sobel_edges.jpg` |
| 2 | Kernel Emboss | Tích chập kernel tùy chỉnh | `emboss.jpg` |
| 3 | So sánh bộ lọc | Grid 4 ảnh | `filter_comparison.jpg` |
| 4 | Lọc trung vị (k=5) | `cv2.medianBlur` | `median_nonlinear.jpg` |
| 5 | Lọc song phương | `cv2.bilateralFilter` | `bilateral_filtered.jpg` |

**Kết luận:** Sobel phát hiện cạnh tốt nhưng nhạy nhiễu. Bilateral filter là lọc phi tuyến mạnh nhất: vừa giảm nhiễu vừa bảo toàn cạnh nhờ kết hợp cả khoảng cách không gian và cường độ.